<a href="https://colab.research.google.com/github/oesquivel81/TopoRAG/blob/main/02_document_ingestion_and_cleaning_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 — Document Ingestion and Cleaning

This notebook prepares the arXiv corpus before chunking and embedding.

Cleaning objectives:

- Preserve mathematical notation.
- Remove extraction artifacts.
- Repair words divided across lines.
- Detect empty or suspicious pages.
- Detect repeated headers and footers.
- Detect exact duplicates.
- Preserve the original content for auditing.

In [1]:
%%capture --no-stderr
%pip install --quiet -U langchain_openai langchain_core langchain_community langchain-tavily  pymupdf pandas

In [2]:
!git clone https://github.com/oesquivel81/TopoRAG.git /content/TopoRAG
%cd /content/TopoRAG
!pip install -q -r /content/TopoRAG/colab_export/requirements.txt

import sys
sys.path.append('/content/TopoRAG/colab_export')

from arxiv_corpus_collector import ArxivCorpusCollector

collector = ArxivCorpusCollector(
    search_queries={
        "differential_geometry": 'cat:math.DG AND (all:"differential geometry" OR all:"Riemannian geometry")',
        "algebraic_topology": 'cat:math.AT AND (all:"algebraic topology" OR all:"homology" OR all:"cohomology")',
        "topological_data_analysis": '(cat:math.AT OR cat:cs.LG OR cat:stat.ML) AND (all:"persistent homology" OR all:"topological data analysis")',
    },
    project_root='/content/TopoRAG',
    max_results_per_query=10,
)

result = collector.run()
corpus_df = result["corpus_df"]

display(corpus_df.head())

Cloning into '/content/TopoRAG'...
remote: Enumerating objects: 28, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (23/23), done.
Receiving objects: 100% (28/28), 51.88 KiB | 3.99 MiB/s, done.
remote: Total 28 (delta 4), reused 7 (delta 1), pack-reused 0 (from 0)
Resolving deltas: 100% (4/4), done.
/content/TopoRAG
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


20:19:29 | INFO | Project root: /content/TopoRAG
20:19:29 | INFO | Raw PDF directory: /content/TopoRAG/data/raw/arxiv
20:19:29 | INFO | Manifest directory: /content/TopoRAG/data/manifests
20:19:29 | INFO | Interim directory: /content/TopoRAG/data/interim
20:19:29 | INFO | Searching topic: differential_geometry
20:19:30 | INFO | Retrieved 10 records for topic: differential_geometry
20:19:30 | INFO | Searching topic: algebraic_topology
20:19:33 | INFO | Retrieved 10 records for topic: algebraic_topology
20:19:33 | INFO | Searching topic: topological_data_analysis
20:19:37 | INFO | Retrieved 10 records for topic: topological_data_analysis
20:19:37 | INFO | PDF 2605.02279: downloaded
20:19:38 | INFO | PDF 2603.13771: downloaded
20:19:39 | INFO | PDF 2601.17166: downloaded
20:19:41 | INFO | PDF 2601.02325: downloaded
20:19:42 | INFO | PDF 2408.05318: downloaded
20:19:43 | INFO | PDF 2305.04281: downloaded
20:19:44 | INFO | PDF 2211.02520: downloaded
20:19:45 | INFO | PDF 2210.02569: downloa

,arxiv_id,source_file,page_number,total_pages,page_content,char_count,word_count,is_empty_raw,extracted_at_utc,arxiv_version,...,query_hit_count,pdf_filename,pdf_relative_path,download_status,download_error,pdf_size_bytes,downloaded_at_utc,extraction_status,extraction_error,extracted_page_count
0,2605.02279,2605.02279.pdf,1,143,Foundations of Riemannian Geometry for Riemann...,2730,338,False,2026-07-14 20:20:10.746802+00:00,1,...,1,2605.02279.pdf,data/raw/arxiv/2605.02279.pdf,downloaded,None,5913737,2026-07-14 20:19:37.800321+00:00,success,None,143
1,2605.02279,2605.02279.pdf,2,143,Foundations of Riemannian Geometry for Riemann...,5641,2025,False,2026-07-14 20:20:10.746802+00:00,1,...,1,2605.02279.pdf,data/raw/arxiv/2605.02279.pdf,downloaded,None,5913737,2026-07-14 20:19:37.800321+00:00,success,None,143
2,2605.02279,2605.02279.pdf,3,143,Foundations of Riemannian Geometry for Riemann...,6176,2309,False,2026-07-14 20:20:10.746802+00:00,1,...,1,2605.02279.pdf,data/raw/arxiv/2605.02279.pdf,downloaded,None,5913737,2026-07-14 20:19:37.800321+00:00,success,None,143
3,2605.02279,2605.02279.pdf,4,143,Foundations of Riemannian Geometry for Riemann...,6587,2276,False,2026-07-14 20:20:10.746802+00:00,1,...,1,2605.02279.pdf,data/raw/arxiv/2605.02279.pdf,downloaded,None,5913737,2026-07-14 20:19:37.800321+00:00,success,None,143
4,2605.02279,2605.02279.pdf,5,143,Foundations of Riemannian Geometry for Riemann...,3410,1132,False,2026-07-14 20:20:10.746802+00:00,1,...,1,2605.02279.pdf,data/raw/arxiv/2605.02279.pdf,downloaded,None,5913737,2026-07-14 20:19:37.800321+00:00,success,None,143


In [3]:
%%capture --no-stderr

%pip install --quiet pandas==2.2.2

In [4]:
from pathlib import Path
import subprocess

PROJECT_ROOT = Path("/content/TopoRAG")

if not (PROJECT_ROOT / ".git").exists():
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/oesquivel81/TopoRAG.git",
            str(PROJECT_ROOT),
        ],
        check=True,
    )
else:
    print("El repositorio ya existe:", PROJECT_ROOT)

El repositorio ya existe: /content/TopoRAG


In [6]:
display(corpus_df.head(3))

,arxiv_id,source_file,page_number,total_pages,page_content,char_count,word_count,is_empty_raw,extracted_at_utc,arxiv_version,...,query_hit_count,pdf_filename,pdf_relative_path,download_status,download_error,pdf_size_bytes,downloaded_at_utc,extraction_status,extraction_error,extracted_page_count
0,2605.02279,2605.02279.pdf,1,143,Foundations of Riemannian Geometry for Riemann...,2730,338,False,2026-07-14 20:35:00.625617+00:00,1,...,1,2605.02279.pdf,data/raw/arxiv/2605.02279.pdf,existing,None,5913737,NaT,success,None,143
1,2605.02279,2605.02279.pdf,2,143,Foundations of Riemannian Geometry for Riemann...,5641,2025,False,2026-07-14 20:35:00.625617+00:00,1,...,1,2605.02279.pdf,data/raw/arxiv/2605.02279.pdf,existing,None,5913737,NaT,success,None,143
2,2605.02279,2605.02279.pdf,3,143,Foundations of Riemannian Geometry for Riemann...,6176,2309,False,2026-07-14 20:35:00.625617+00:00,1,...,1,2605.02279.pdf,data/raw/arxiv/2605.02279.pdf,existing,None,5913737,NaT,success,None,143


In [7]:
corpus_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1325 entries, 0 to 1324
Data columns (total 37 columns):
 #   Column                Non-Null Count  Dtype              
---  ------                --------------  -----              
 0   arxiv_id              1325 non-null   str                
 1   source_file           1325 non-null   str                
 2   page_number           1325 non-null   int64              
 3   total_pages           1325 non-null   int64              
 4   page_content          1325 non-null   str                
 5   char_count            1325 non-null   int64              
 6   word_count            1325 non-null   int64              
 7   is_empty_raw          1325 non-null   bool               
 8   extracted_at_utc      1325 non-null   datetime64[us, UTC]
 9   arxiv_version         1325 non-null   int64              
 10  entry_id              1325 non-null   str                
 11  pdf_url               1325 non-null   str                
 12  title            

In [8]:
null_report = (
    corpus_df.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("null_values")
)

display(null_report)

,null_values
extraction_error,1325
download_error,1325
downloaded_at_utc,1325
doi,1174
journal_ref,962
comment,63
page_content,0
page_number,0
source_file,0
arxiv_id,0


In [12]:
corpus_summary = {
    "rows_pages": len(corpus_df),
    "unique_articles": corpus_df["arxiv_id"].nunique(),
    "unique_pdf_files": corpus_df["source_file"].nunique(),
    "empty_pages": int(corpus_df["is_empty_raw"].sum()),
    "total_characters": int(corpus_df["char_count"].sum()),
    "total_words": int(corpus_df["word_count"].sum()),
}

corpus_summary

{'rows_pages': 1325,
 'unique_articles': 30,
 'unique_pdf_files': 30,
 'empty_pages': 0,
 'total_characters': 3197650,
 'total_words': 572290}

In [13]:
articles_df = (
    corpus_df
    .groupby(
        ["arxiv_id", "title", "source_file"],
        as_index=False,
    )
    .agg(
        extracted_pages=("page_number", "nunique"),
        reported_total_pages=("total_pages", "max"),
        total_characters=("char_count", "sum"),
        total_words=("word_count", "sum"),
        topics=("topics", "first"),
    )
    .sort_values("extracted_pages", ascending=False)
)

display(articles_df.head(20))

,arxiv_id,title,source_file,extracted_pages,reported_total_pages,total_characters,total_words,topics
22,2601.02325,"A translation of ""What is differential geometr...",2601.02325.pdf,214,214,390354,66781,[differential_geometry]
0,0808.1395,Algebraic topology from geometric viewpoint,0808.1395.pdf,168,168,427542,67927,[algebraic_topology]
27,math/0504358,Discrete differential geometry. Consistency as...,math__0504358.pdf,157,157,276826,49634,[differential_geometry]
25,2605.02279,Foundations of Riemannian Geometry for Riemann...,2605.02279.pdf,143,143,507941,93346,[differential_geometry]
5,1206.3093,Sub-riemannian geometry from intrinsic viewpoint,1206.3093.pdf,58,58,142891,28175,[differential_geometry]
18,2210.02569,"Semi-coarse Spaces, Homotopy and Homology",2210.02569.pdf,56,56,141647,29237,[algebraic_topology]
21,2408.05318,Recent Advances in Metallic Riemannian Geometr...,2408.05318.pdf,47,47,101341,18365,[differential_geometry]
2,1009.5156,Behavior of Quillen (co)homology with respect ...,1009.5156.pdf,38,38,99869,18023,[algebraic_topology]
6,1304.7846,Algebraic Topology,1304.7846.pdf,36,36,85522,15336,[algebraic_topology]
10,1708.07390,Stratifying multiparameter persistent homology,1708.07390.pdf,33,33,85867,15919,[topological_data_analysis]


In [14]:
print("Estados de descarga:")
display(
    corpus_df["download_status"]
    .value_counts(dropna=False)
    .to_frame("pages")
)

print("\nEstados de extracción:")
display(
    corpus_df["extraction_status"]
    .value_counts(dropna=False)
    .to_frame("pages")
)

Estados de descarga:


,pages
download_status,
existing,1325



Estados de extracción:


,pages
extraction_status,
success,1325


In [15]:
empty_pages_df = corpus_df[
    corpus_df["is_empty_raw"]
].copy()

print("Páginas vacías:", len(empty_pages_df))

display(
    empty_pages_df[
        [
            "arxiv_id",
            "title",
            "source_file",
            "page_number",
            "char_count",
            "word_count",
        ]
    ].head(30)
)

Páginas vacías: 0


,arxiv_id,title,source_file,page_number,char_count,word_count


In [16]:
working_df = corpus_df.copy()

working_df["raw_text"] = (
    working_df["page_content"]
    .fillna("")
    .astype(str)
)

In [17]:
import re
import unicodedata


def clean_mathematical_text(text: str) -> str:
    if not text:
        return ""

    # Conserva la representación matemática mejor que una
    # normalización agresiva como NFKC.
    text = unicodedata.normalize("NFC", text)

    # Eliminar bytes nulos.
    text = text.replace("\x00", "")

    # Eliminar caracteres de control, conservando saltos y tabulaciones.
    text = "".join(
        character
        for character in text
        if character in "\n\t"
        or unicodedata.category(character)[0] != "C"
    )

    # Unir palabras divididas al final de una línea.
    # Ejemplo: "homo-\nlogy" -> "homology".
    # Solo se une cuando la segunda parte empieza en minúscula.
    text = re.sub(
        r"(?<=[A-Za-zÁÉÍÓÚáéíóúñÑ])-\s*\n\s*"
        r"(?=[a-záéíóúñ])",
        "",
        text,
    )

    # Reducir espacios y tabuladores repetidos.
    text = re.sub(r"[ \t]+", " ", text)

    # Limpiar espacios alrededor de saltos de línea.
    text = re.sub(r" *\n *", "\n", text)

    # Conservar párrafos, pero eliminar saltos excesivos.
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Eliminar espacios antes de puntuación.
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)

    return text.strip()

In [18]:
working_df["basic_clean_text"] = (
    working_df["raw_text"]
    .apply(clean_mathematical_text)
)

working_df["basic_clean_char_count"] = (
    working_df["basic_clean_text"]
    .str.len()
)

working_df["basic_clean_word_count"] = (
    working_df["basic_clean_text"]
    .str.split()
    .str.len()
)

In [19]:
row_index = 10

print("DOCUMENTO:", working_df.loc[row_index, "title"])
print("PÁGINA:", working_df.loc[row_index, "page_number"])

print("\nTEXTO ORIGINAL")
print("=" * 100)
print(working_df.loc[row_index, "raw_text"][:3500])

print("\nTEXTO CON LIMPIEZA BÁSICA")
print("=" * 100)
print(working_df.loc[row_index, "basic_clean_text"][:3500])

DOCUMENTO: Foundations of Riemannian Geometry for Riemannian Optimization: A Monograph with Detailed Derivations
PÁGINA: 11

TEXTO ORIGINAL
Foundations of Riemannian Geometry for Riemannian Optimization: A Monograph with Detailed Derivations
11
Definition 14 (Point on a manifold). Let M be a manifold.
A point on M is simply an element of the set M. In other
words, if p ∈M, then p is a point of the manifold.
In differential geometry, the word “point” refers to a loca-
tion on the manifold itself, whereas tangent vectors, cotan-
gent vectors, and tensors are objects attached to that point.
Definition 15 (Tangent space on a smooth or Riemannian
manifold). Let M be a smooth manifold and let p ∈M.
The tangent space TpM is the vector space consisting of
all tangent vectors at point p, which represent possible di-
rections in which one can pass through p along smooth
curves on the manifold.
Intuitively, the tangent space is a linear approximation of
the manifold around the point p. For a Riem

In [20]:
from collections import Counter
from math import ceil


def get_non_empty_lines(text: str) -> list[str]:
    return [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]


def normalize_boundary_line(line: str) -> str:
    line = line.strip().lower()
    line = re.sub(r"\s+", " ", line)
    return line


def is_standalone_page_number(line: str) -> bool:
    return bool(
        re.fullmatch(
            r"(?:page\s*)?\d{1,4}",
            line.strip(),
            flags=re.IGNORECASE,
        )
    )

In [21]:
def detect_repeated_boundaries(
    dataframe,
    boundary_depth: int = 2,
    minimum_ratio: float = 0.30,
) -> dict:
    boundaries_by_article = {}

    for arxiv_id, group in dataframe.groupby("arxiv_id"):
        header_counter = Counter()
        footer_counter = Counter()

        page_count = len(group)

        for text in group["basic_clean_text"]:
            lines = get_non_empty_lines(text)

            if not lines:
                continue

            header_lines = lines[:boundary_depth]
            footer_lines = lines[-boundary_depth:]

            header_counter.update(
                normalize_boundary_line(line)
                for line in header_lines
                if len(line.strip()) >= 2
            )

            footer_counter.update(
                normalize_boundary_line(line)
                for line in footer_lines
                if len(line.strip()) >= 2
            )

        minimum_repetitions = max(
            3,
            ceil(page_count * minimum_ratio),
        )

        repeated_headers = {
            line
            for line, count in header_counter.items()
            if count >= minimum_repetitions
        }

        repeated_footers = {
            line
            for line, count in footer_counter.items()
            if count >= minimum_repetitions
        }

        boundaries_by_article[arxiv_id] = {
            "headers": repeated_headers,
            "footers": repeated_footers,
            "minimum_repetitions": minimum_repetitions,
        }

    return boundaries_by_article

In [22]:
boundaries_by_article = detect_repeated_boundaries(
    working_df,
    boundary_depth=2,
    minimum_ratio=0.30,
)

In [23]:
for arxiv_id, boundaries in list(
    boundaries_by_article.items()
)[:5]:
    print("=" * 100)
    print("arXiv:", arxiv_id)
    print("Headers:", boundaries["headers"])
    print("Footers:", boundaries["footers"])

arXiv: 0808.1395
Headers: set()
Footers: set()
arXiv: 0907.1283
Headers: set()
Footers: set()
arXiv: 1009.5156
Headers: {'martin frankland', 'behavior of quillen (co)homology with respect to adjunctions'}
Footers: set()
arXiv: 1105.6294
Headers: set()
Footers: set()
arXiv: 1202.6132
Headers: set()
Footers: set()


In [24]:
def remove_repeated_boundaries(
    text: str,
    arxiv_id: str,
    boundaries: dict,
    boundary_depth: int = 2,
) -> str:
    lines = get_non_empty_lines(text)

    if not lines:
        return ""

    article_boundaries = boundaries.get(
        arxiv_id,
        {
            "headers": set(),
            "footers": set(),
        },
    )

    repeated_headers = article_boundaries["headers"]
    repeated_footers = article_boundaries["footers"]

    cleaned_lines = lines.copy()

    # Revisar solamente las primeras líneas.
    for _ in range(min(boundary_depth, len(cleaned_lines))):
        if not cleaned_lines:
            break

        first_line = cleaned_lines[0]
        normalized = normalize_boundary_line(first_line)

        if (
            normalized in repeated_headers
            or is_standalone_page_number(first_line)
        ):
            cleaned_lines.pop(0)
        else:
            break

    # Revisar solamente las últimas líneas.
    for _ in range(min(boundary_depth, len(cleaned_lines))):
        if not cleaned_lines:
            break

        last_line = cleaned_lines[-1]
        normalized = normalize_boundary_line(last_line)

        if (
            normalized in repeated_footers
            or is_standalone_page_number(last_line)
        ):
            cleaned_lines.pop()
        else:
            break

    return "\n".join(cleaned_lines).strip()

In [25]:
working_df["clean_text"] = working_df.apply(
    lambda row: remove_repeated_boundaries(
        text=row["basic_clean_text"],
        arxiv_id=row["arxiv_id"],
        boundaries=boundaries_by_article,
        boundary_depth=2,
    ),
    axis=1,
)

working_df["clean_char_count"] = (
    working_df["clean_text"]
    .str.len()
)

working_df["clean_word_count"] = (
    working_df["clean_text"]
    .str.split()
    .str.len()
)

In [26]:
working_df["removed_characters"] = (
    working_df["char_count"]
    - working_df["clean_char_count"]
)

working_df["removed_percentage"] = (
    working_df["removed_characters"]
    .div(working_df["char_count"].replace(0, 1))
    .mul(100)
    .round(2)
)

display(
    working_df[
        [
            "char_count",
            "clean_char_count",
            "removed_characters",
            "removed_percentage",
        ]
    ].describe()
)

,char_count,clean_char_count,removed_characters,removed_percentage
count,1325.000000,1325.000000,1325.000000,1325.000000
mean,2413.320755,2364.480000,48.840755,1.972068
std,936.937406,904.231378,116.850026,4.111221
min,2.000000,0.000000,1.000000,0.030000
25%,1794.000000,1768.000000,8.000000,0.390000
50%,2358.000000,2311.000000,17.000000,0.840000
75%,2963.000000,2906.000000,61.000000,2.530000
max,6587.000000,5770.000000,2093.000000,100.000000


In [27]:
suspicious_cleaning_df = working_df[
    working_df["removed_percentage"] > 25
][
    [
        "arxiv_id",
        "title",
        "page_number",
        "char_count",
        "clean_char_count",
        "removed_percentage",
    ]
].sort_values(
    "removed_percentage",
    ascending=False,
)

display(suspicious_cleaning_df.head(30))

,arxiv_id,title,page_number,char_count,clean_char_count,removed_percentage
1158,math/0504358,Discrete differential geometry. Consistency as...,2,2,0,100.00
2,2605.02279,Foundations of Riemannian Geometry for Riemann...,3,6176,4083,33.89
1,2605.02279,Foundations of Riemannian Geometry for Riemann...,2,5641,3838,31.96
3,2605.02279,Foundations of Riemannian Geometry for Riemann...,4,6587,4593,30.27
4,2605.02279,Foundations of Riemannian Geometry for Riemann...,5,3410,2392,29.85
1159,math/0504358,Discrete differential geometry. Consistency as...,3,1667,1193,28.43
773,1304.7846,Algebraic Topology,2,1571,1132,27.94
590,2209.05134,On topological data analysis for structural dy...,25,3875,2818,27.28


In [28]:
def classify_page_quality(text: str) -> str:
    characters = len(text)
    words = len(text.split())

    if characters == 0:
        return "empty"

    if characters < 100 or words < 15:
        return "suspicious_short"

    if characters < 300 or words < 40:
        return "short"

    return "usable"

In [29]:
working_df["quality"] = (
    working_df["clean_text"]
    .apply(classify_page_quality)
)

quality_summary = (
    working_df["quality"]
    .value_counts()
    .rename_axis("quality")
    .reset_index(name="pages")
)

display(quality_summary)

,quality,pages
0,usable,1305
1,short,10
2,suspicious_short,9
3,empty,1


In [30]:
display(
    working_df[
        working_df["quality"].isin(
            ["empty", "suspicious_short"]
        )
    ][
        [
            "arxiv_id",
            "title",
            "page_number",
            "total_pages",
            "clean_char_count",
            "clean_word_count",
            "quality",
        ]
    ].head(50)
)

,arxiv_id,title,page_number,total_pages,clean_char_count,clean_word_count,quality
191,2601.02325,"A translation of ""What is differential geometr...",8,214,11,1,suspicious_short
192,2601.02325,"A translation of ""What is differential geometr...",9,214,14,3,suspicious_short
257,2601.02325,"A translation of ""What is differential geometr...",74,214,23,4,suspicious_short
258,2601.02325,"A translation of ""What is differential geometr...",75,214,20,3,suspicious_short
300,2601.02325,"A translation of ""What is differential geometr...",117,214,23,3,suspicious_short
1158,math/0504358,Discrete differential geometry. Consistency as...,2,157,0,0,empty
1168,math/0504358,Discrete differential geometry. Consistency as...,12,157,23,3,suspicious_short
1194,math/0504358,Discrete differential geometry. Consistency as...,38,157,42,5,suspicious_short
1242,math/0504358,Discrete differential geometry. Consistency as...,86,157,24,3,suspicious_short
1304,math/0504358,Discrete differential geometry. Consistency as...,148,157,37,5,suspicious_short


In [31]:
suspicious_sample = working_df[
    working_df["quality"] == "suspicious_short"
].head(5)

for _, row in suspicious_sample.iterrows():
    print("=" * 100)
    print(row["title"])
    print("Página:", row["page_number"])
    print()
    print(row["clean_text"])

A translation of "What is differential geometry: curves and surfaces"
Página: 8

ПРЕДИСЛОВИЕ
A translation of "What is differential geometry: curves and surfaces"
Página: 9

Часть I
Кривые
A translation of "What is differential geometry: curves and surfaces"
Página: 74

ГЛАВА 7. ОПОРНЫЕ КРИВЫЕ
A translation of "What is differential geometry: curves and surfaces"
Página: 75

Часть II
Поверхности
A translation of "What is differential geometry: curves and surfaces"
Página: 117

Часть III
Геодезические


In [34]:
import hashlib

def create_content_hash(text: str) -> str:
    if not text:
        return ""
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

working_df["content_hash"] = (
    working_df["clean_text"]
    .apply(create_content_hash)
)

duplicate_mask = (
    working_df["content_hash"].ne("")
    & working_df.duplicated(
        subset=["content_hash"],
        keep="first",
    )
)

working_df["is_exact_duplicate"] = duplicate_mask

print(
    "Páginas duplicadas exactas:",
    int(working_df["is_exact_duplicate"].sum()),
)

Páginas duplicadas exactas: 0


In [35]:
display(
    working_df[
        working_df["is_exact_duplicate"]
    ][
        [
            "arxiv_id",
            "title",
            "page_number",
            "source_file",
            "content_hash",
        ]
    ].head(30)
)

,arxiv_id,title,page_number,source_file,content_hash


In [36]:
clean_corpus_df = (
    working_df[
        (working_df["quality"] != "empty")
        & (~working_df["is_exact_duplicate"])
    ]
    .copy()
    .reset_index(drop=True)
)

print("Páginas iniciales:", len(corpus_df))
print("Páginas limpias:", len(clean_corpus_df))
print(
    "Artículos conservados:",
    clean_corpus_df["arxiv_id"].nunique(),
)

Páginas iniciales: 1325
Páginas limpias: 1324
Artículos conservados: 30


In [37]:
clean_corpus_df["raw_page_content"] = (
    clean_corpus_df["page_content"]
)

clean_corpus_df["page_content"] = (
    clean_corpus_df["clean_text"]
)

In [38]:
from pathlib import Path

PROCESSED_DIR = Path(
    "/content/TopoRAG/data/processed"
)

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

clean_parquet_path = (
    PROCESSED_DIR / "arxiv_pages_clean.parquet"
)

clean_jsonl_path = (
    PROCESSED_DIR / "arxiv_pages_clean.jsonl"
)

quality_report_path = (
    PROCESSED_DIR / "page_quality_report.csv"
)

In [39]:
clean_corpus_df.to_parquet(
    clean_parquet_path,
    index=False,
)

clean_corpus_df.to_json(
    clean_jsonl_path,
    orient="records",
    lines=True,
    force_ascii=False,
    date_format="iso",
)

working_df[
    [
        "arxiv_id",
        "title",
        "source_file",
        "page_number",
        "total_pages",
        "char_count",
        "word_count",
        "clean_char_count",
        "clean_word_count",
        "removed_percentage",
        "quality",
        "is_exact_duplicate",
        "content_hash",
    ]
].to_csv(
    quality_report_path,
    index=False,
    encoding="utf-8",
)

print("Parquet:", clean_parquet_path)
print("JSONL:", clean_jsonl_path)
print("Reporte:", quality_report_path)

Parquet: /content/TopoRAG/data/processed/arxiv_pages_clean.parquet
JSONL: /content/TopoRAG/data/processed/arxiv_pages_clean.jsonl
Reporte: /content/TopoRAG/data/processed/page_quality_report.csv
